# Step 2: Data Cleaning

## Objective

The purpose of this notebook is to prepare the DataCo dataset for business analysis, SQL, and the final dashboard.

The raw dataset will not be changed. A separate working copy will be cleaned so the original data remains available.

This notebook will:

1. Load the raw dataset
2. Create a working copy
3. Decide which columns are needed
4. Remove sensitive and unnecessary columns
5. Convert the date columns
6. Check invalid or unusual values
7. Combine item rows into one row per order
8. Validate and export the cleaned order-level dataset


## 2.1 Load the Raw Dataset

The dataset must be loaded again because variables created in Notebook 1 do not automatically exist in Notebook 2.

`latin-1` is used because Notebook 1 showed that this file requires that character encoding. Encoding controls how Python reads characters; it is not related to file size.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)

# Locate the project folder
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

raw_data_file = RAW_DATA_DIR / "DataCoSupplyChainDataset.csv"

# Load the complete raw dataset
raw_df = pd.read_csv(
    raw_data_file,
    encoding="latin-1",
    low_memory=False
)

print("Dataset loaded successfully.")
print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])


Dataset loaded successfully.
Rows: 180519
Columns: 53


## 2.2 Create a Working Copy

`raw_df` will remain unchanged. `clean_df` is a separate copy that can be changed safely during cleaning.


In [2]:
clean_df = raw_df.copy()

print("Raw dataset shape:", raw_df.shape)
print("Working copy shape:", clean_df.shape)


Raw dataset shape: (180519, 53)
Working copy shape: (180519, 53)


## 2.3 Classify the Columns

Before removing anything, the columns are divided into four groups:

- **Order-level columns:** one value can be kept for each order.
- **Item-level columns to aggregate:** values must be counted or summed for each order.
- **Sensitive columns:** personal information that is not needed for the analysis.
- **Unnecessary columns:** fields that do not help answer the business question or duplicate other information.

This classification prevents useful fields from being removed accidentally.


In [3]:
order_level_columns = [
    "Order Id", "Order Customer Id", "order date (DateOrders)",
    "shipping date (DateOrders)", "Type", "Customer Segment",
    "Market", "Order Region", "Order Country", "Order State",
    "Order City", "Shipping Mode", "Order Status", "Delivery Status",
    "Late_delivery_risk", "Days for shipping (real)",
    "Days for shipment (scheduled)"
]

aggregate_columns = [
    "Order Item Id", "Order Item Quantity", "Product Card Id",
    "Product Category Id", "Sales", "Order Item Total",
    "Order Item Discount", "Order Profit Per Order"
]

sensitive_columns = [
    "Customer Email", "Customer Fname", "Customer Lname",
    "Customer Password", "Customer Street"
]

unnecessary_columns = [
    "Benefit per order", "Sales per customer", "Category Id",
    "Category Name", "Customer City", "Customer Country", "Customer Id",
    "Customer State", "Customer Zipcode", "Department Id", "Department Name",
    "Latitude", "Longitude", "Order Item Cardprod Id",
    "Order Item Discount Rate", "Order Item Product Price",
    "Order Item Profit Ratio", "Order Zipcode", "Product Description",
    "Product Image", "Product Name", "Product Price", "Product Status"
]

classified_columns = (
    order_level_columns
    + aggregate_columns
    + sensitive_columns
    + unnecessary_columns
)

unclassified_columns = sorted(set(clean_df.columns) - set(classified_columns))
classified_more_than_once = sorted({
    column for column in classified_columns
    if classified_columns.count(column) > 1
})

print("Dataset columns:", len(clean_df.columns))
print("Classified columns:", len(set(classified_columns)))
print("Unclassified columns:", unclassified_columns)
print("Columns classified more than once:", classified_more_than_once)


Dataset columns: 53
Classified columns: 53
Unclassified columns: []
Columns classified more than once: []


### Why Some Columns Are Not Kept

- Customer names, email, password, and street are removed for privacy.
- `Product Description` is completely empty.
- `Order Zipcode` is mostly empty and broader destination fields are available.
- Product and category details can differ between items in one order. Instead of choosing one product arbitrarily, the order-level dataset will count the number of products and categories.
- `Order Item Total`, quantity, discount, sales, and profit will be summed because they are item-level values.


## 2.4 Review Missing Values

Missing values are checked before aggregation. Only columns containing at least one missing value are displayed.

The goal is not to fill every empty cell. If a column is unnecessary, removing the column is more honest than inventing values.


In [4]:
missing_summary = pd.DataFrame({
    "Missing Values": clean_df.isna().sum(),
    "Missing Percentage": (clean_df.isna().mean() * 100).round(4)
})

missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

display(missing_summary)


,Missing Values,Missing Percentage
Product Description,180519,100.0000
Order Zipcode,155679,86.2397
Customer Lname,8,0.0044
Customer Zipcode,3,0.0017


### Missing-Value Decision

The columns containing missing values are not required in the final order-level dataset:

- `Product Description` and `Order Zipcode` are unnecessary.
- Customer last name and customer zipcode are not needed and will be excluded for privacy or business relevance.

Therefore, no missing values need to be filled. No artificial values such as `Unknown` or averages will be added.


## 2.5 Convert the Date Columns

The order and shipping dates were originally read as text (`object`). They must be converted to `datetime` so that Python can sort dates, calculate time periods, and create monthly trends.

`errors="coerce"` changes an invalid date into a missing date instead of stopping the notebook. The next check counts whether any conversion failed.


In [5]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for column in date_columns:
    clean_df[column] = pd.to_datetime(
        clean_df[column],
        errors="coerce"
    )

print(clean_df[date_columns].dtypes)
print("\nMissing dates after conversion:")
print(clean_df[date_columns].isna().sum())
print("\nOrder date range:", clean_df["order date (DateOrders)"].min(), "to", clean_df["order date (DateOrders)"].max())
print("Shipping date range:", clean_df["shipping date (DateOrders)"].min(), "to", clean_df["shipping date (DateOrders)"].max())


order date (DateOrders)       datetime64[ns]
shipping date (DateOrders)    datetime64[ns]
dtype: object

Missing dates after conversion:
order date (DateOrders)       0
shipping date (DateOrders)    0
dtype: int64

Order date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00
Shipping date range: 2015-01-03 00:00:00 to 2018-02-06 22:14:00


## 2.6 Check Important Values

Before creating the order-level table, important operational values are checked for obvious problems.

The checks look for:

- Negative shipping durations
- Shipping dates earlier than order dates
- Unexpected delivery-status categories
- Unexpected late-delivery indicators

An unusual value is not automatically deleted. It must first be understood.


In [6]:
negative_actual_days = (clean_df["Days for shipping (real)"] < 0).sum()
negative_scheduled_days = (clean_df["Days for shipment (scheduled)"] < 0).sum()
shipping_before_order = (
    clean_df["shipping date (DateOrders)"]
    < clean_df["order date (DateOrders)"]
).sum()

print("Negative actual shipping days:", negative_actual_days)
print("Negative scheduled shipping days:", negative_scheduled_days)
print("Shipping dates before order dates:", shipping_before_order)
print("\nDelivery Status values:")
print(sorted(clean_df["Delivery Status"].dropna().unique()))
print("\nLate-delivery-risk values:")
print(sorted(clean_df["Late_delivery_risk"].dropna().unique()))


Negative actual shipping days: 0
Negative scheduled shipping days: 0
Shipping dates before order dates: 0

Delivery Status values:
['Advance shipping', 'Late delivery', 'Shipping canceled', 'Shipping on time']

Late-delivery-risk values:
[np.int64(0), np.int64(1)]


## 2.7 Create the Order-Level Dataset

Notebook 1 showed that the raw data contains one row per order-item line. The final analytical dataset needs one row per order.

The aggregation uses two simple rules:

- `first`: keep one value when Notebook 1 confirmed the field is consistent within an order.
- `sum`, `count`, or `nunique`: combine item-level information for the complete order.

For example, `order_value` is the sum of all item totals in the order. Keeping only the first item total would understate the order value.


In [7]:
orders_clean = (
    clean_df.groupby("Order Id", as_index=False)
    .agg(
        customer_id=("Order Customer Id", "first"),
        order_date=("order date (DateOrders)", "first"),
        shipping_date=("shipping date (DateOrders)", "first"),
        transaction_type=("Type", "first"),
        customer_segment=("Customer Segment", "first"),
        market=("Market", "first"),
        order_region=("Order Region", "first"),
        order_country=("Order Country", "first"),
        order_state=("Order State", "first"),
        order_city=("Order City", "first"),
        shipping_mode=("Shipping Mode", "first"),
        order_status=("Order Status", "first"),
        delivery_status=("Delivery Status", "first"),
        late_delivery_risk=("Late_delivery_risk", "first"),
        actual_shipping_days=("Days for shipping (real)", "first"),
        scheduled_shipping_days=("Days for shipment (scheduled)", "first"),
        item_line_count=("Order Item Id", "count"),
        total_quantity=("Order Item Quantity", "sum"),
        unique_product_count=("Product Card Id", "nunique"),
        unique_category_count=("Product Category Id", "nunique"),
        gross_sales=("Sales", "sum"),
        order_value=("Order Item Total", "sum"),
        total_discount=("Order Item Discount", "sum"),
        order_profit=("Order Profit Per Order", "sum")
    )
    .rename(columns={"Order Id": "order_id"})
)

print("Raw item-level rows:", len(clean_df))
print("Clean order-level rows:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())


Raw item-level rows: 180519
Clean order-level rows: 65752
Unique order IDs: 65752


## 2.8 Create Useful Analytical Columns

New columns are created from existing data:

- `delay_days`: actual shipping days minus scheduled shipping days.
- `is_delivery_exception`: 1 for late or canceled shipping, otherwise 0.
- `exception_type`: a readable label for the exception.
- Date parts: year, month, and year-month for trend analysis.

Advance shipping is kept as a delivery status, but it is not automatically labelled as a harmful exception because the dataset does not show whether early arrival caused a business problem.


In [8]:
orders_clean["delay_days"] = (
    orders_clean["actual_shipping_days"]
    - orders_clean["scheduled_shipping_days"]
)

orders_clean["is_delivery_exception"] = (
    orders_clean["delivery_status"].isin([
        "Late delivery",
        "Shipping canceled"
    ])
).astype(int)

orders_clean["exception_type"] = orders_clean["delivery_status"].map({
    "Late delivery": "Late Delivery",
    "Shipping canceled": "Shipping Canceled",
    "Advance shipping": "No Delivery Exception",
    "Shipping on time": "No Delivery Exception"
})

orders_clean["order_year"] = orders_clean["order_date"].dt.year
orders_clean["order_month"] = orders_clean["order_date"].dt.month
orders_clean["order_year_month"] = orders_clean["order_date"].dt.to_period("M").astype(str)

display(orders_clean.head())


,order_id,customer_id,order_date,shipping_date,transaction_type,customer_segment,market,order_region,order_country,order_state,order_city,shipping_mode,order_status,delivery_status,late_delivery_risk,actual_shipping_days,scheduled_shipping_days,item_line_count,total_quantity,unique_product_count,unique_category_count,gross_sales,order_value,total_discount,order_profit,delay_days,is_delivery_exception,exception_type,order_year,order_month,order_year_month
0,1,11599,2015-01-01 00:00:00,2015-01-03 00:00:00,CASH,Consumer,LATAM,Central America,México,Distrito Federal,Mexico City,Standard Class,CLOSED,Advance shipping,0,2,4,1,1,1,1,299.980011,239.979996,60.000000,88.790001,-2,0,No Delivery Exception,2015,1,2015-01
1,2,256,2015-01-01 00:21:00,2015-01-04 00:21:00,PAYMENT,Consumer,LATAM,South America,Colombia,Risaralda,Dos Quebradas,Standard Class,PENDING_PAYMENT,Advance shipping,0,3,4,3,7,3,3,579.980011,529.380005,50.600000,195.900002,-1,0,No Delivery Exception,2015,1,2015-01
2,4,8827,2015-01-01 01:03:00,2015-01-06 01:03:00,CASH,Home Office,LATAM,South America,Colombia,Risaralda,Dos Quebradas,Standard Class,CLOSED,Late delivery,1,5,4,4,14,4,4,699.850010,620.870014,78.980000,124.090000,1,1,Late Delivery,2015,1,2015-01
3,5,11318,2015-01-01 01:24:00,2015-01-07 01:24:00,DEBIT,Consumer,LATAM,South America,Colombia,Risaralda,Dos Quebradas,Standard Class,COMPLETE,Late delivery,1,6,4,5,10,4,4,1129.860039,987.070007,142.789999,390.089995,2,1,Late Delivery,2015,1,2015-01
4,7,4530,2015-01-01 02:06:00,2015-01-04 02:06:00,DEBIT,Consumer,LATAM,South America,Brasil,São Paulo,São Paulo,Second Class,COMPLETE,Late delivery,1,3,2,3,7,3,3,579.920013,525.520004,54.400000,203.929998,1,1,Late Delivery,2015,1,2015-01


## 2.9 Validate the Cleaned Dataset

Validation confirms that the cleaning process produced the expected result.

The cleaned table should have:

- 65,752 rows
- 65,752 unique order IDs
- No duplicate order IDs
- No missing values in the required operational fields
- The same late and canceled counts found in Notebook 1


In [9]:
required_columns = [
    "order_id", "order_date", "shipping_date", "delivery_status",
    "shipping_mode", "market", "order_region", "order_country",
    "actual_shipping_days", "scheduled_shipping_days", "order_value"
]

validation_results = {
    "Total rows": len(orders_clean),
    "Unique order IDs": orders_clean["order_id"].nunique(),
    "Duplicate order IDs": orders_clean["order_id"].duplicated().sum(),
    "Missing required values": orders_clean[required_columns].isna().sum().sum(),
    "Late delivery orders": (orders_clean["delivery_status"] == "Late delivery").sum(),
    "Canceled shipping orders": (orders_clean["delivery_status"] == "Shipping canceled").sum()
}

for check, result in validation_results.items():
    print(f"{check}: {result}")


Total rows: 65752
Unique order IDs: 65752
Duplicate order IDs: 0
Missing required values: 0
Late delivery orders: 36048
Canceled shipping orders: 2855


## 2.10 Export the Cleaned Dataset

The cleaned order-level dataset is saved as a CSV file for SQL and Tableau/Power BI.

The `data/processed` folder is excluded by `.gitignore`, so the row-level dataset will remain local and will not be uploaded to GitHub.


In [10]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_DATA_DIR / "dataco_orders_clean.csv"
orders_clean.to_csv(output_file, index=False)

print("Cleaned dataset exported to:", output_file)
print("Exported rows:", len(orders_clean))
print("Exported columns:", len(orders_clean.columns))


Cleaned dataset exported to: /Users/iffah/supply-chain-exception-control-tower/data/processed/dataco_orders_clean.csv
Exported rows: 65752
Exported columns: 31


## Step 2 Conclusion

The raw order-item dataset was cleaned and transformed into an order-level analytical dataset.

Sensitive customer information, empty fields, and unnecessary product-level columns were excluded. Item-level quantity and financial values were aggregated for each order, while consistent shipping and destination fields were kept once.

The cleaned dataset contains one row for each of the 65,752 orders and is ready for exploratory analysis, SQL, and dashboard development.
